In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
import jno

PATH = "/home/users/armbrust/projects/jno/runs"
DATA_PATH = "/home/users/armbrust/projects/jno/examples/poseidon"
SAMPLES = 256

# Neural Operators
fno = jno.core.load(f'{PATH}/fno_data_{SAMPLES}/crux.pkl')
unet = jno.core.load(f'{PATH}/unet_data_{SAMPLES}/crux.pkl')
cno = jno.core.load(f'{PATH}/cno_data_{SAMPLES}/crux.pkl')
pit = jno.core.load(f'{PATH}/pit_data_{SAMPLES}/crux.pkl')
pointnet = jno.core.load(f'{PATH}/pointnet_data_{SAMPLES}/crux.pkl')

# Foundation Models
poseidonT = jno.core.load(f'{PATH}/poseidonT_data_{SAMPLES}/crux.pkl')
poseidonB = jno.core.load(f'{PATH}/poseidonB_data_{SAMPLES}/crux.pkl')
poseidonL = jno.core.load(f'{PATH}/poseidonL_data_{SAMPLES}/crux.pkl')

morphTi = jno.core.load(f'{PATH}/morphTi_data_{SAMPLES}/crux.pkl')
morphS = jno.core.load(f'{PATH}/morphS_data_{SAMPLES}/crux.pkl')
morphM = jno.core.load(f'{PATH}/morphM_data_{SAMPLES}/crux.pkl')
morphL = jno.core.load(f'{PATH}/morphL_data_{SAMPLES}/crux.pkl')

mppTi = jno.core.load(f'{PATH}/mppTi_data_{SAMPLES}/crux.pkl')
mppS = jno.core.load(f'{PATH}/mppS_data_{SAMPLES}/crux.pkl')
mppB = jno.core.load(f'{PATH}/mppB_data_{SAMPLES}/crux.pkl')
mppL = jno.core.load(f'{PATH}/mppL_data_{SAMPLES}/crux.pkl')

pdeformer2_small = jno.core.load(f'{PATH}/pdeformer2_small_data_{SAMPLES}/crux.pkl')
pdeformer2_fast = jno.core.load(f'{PATH}/pdeformer2_fast_data_{SAMPLES}/crux.pkl')
pdeformer2_base = jno.core.load(f'{PATH}/pdeformer2_base_data_{SAMPLES}/crux.pkl')


In [ ]:
models = {'fno':fno, 'unet':unet, 'cno':cno, 'pit':pit, 'pointnet':pointnet,
          'poseidonT':poseidonT, 'poseidonB':poseidonB, 'poseidonL':poseidonL,
          'morphTi':morphTi, 'morphS':morphS, 'morphM':morphM, 'morphL':morphL, 
          'mppTi':mppTi, 'mppS':mppS, 'mppB':mppB, 'mppL':mppL,
          'pdeformer2_small':pdeformer2_small, "pdeformer2_fast":pdeformer2_fast, "pdeformer2_base":pdeformer2_base}

names = list(models.keys())


In [ ]:
#models_lora = {}
for l in [0, 8, 16, 32]:
    for SAMPLES in [4, 16, 32, 64, 128, 256, 512, 1024]:
        for type in ['data', 'both']:
            for mo in ['poseidonT', 'fno']:
                if l != 0:
                    name = f'{mo}_{type}_{SAMPLES}_lora{l}'
                else:
                    name = f'{mo}_{type}_{SAMPLES}'
                if name not in models_lora.keys() or name == 'fno_both_256':
                    try:
                        models_lora[name] = jno.core.load(f'{PATH}/{name}/crux.pkl')
                    except:
                        continue

names_lora = list(models_lora.keys())

In [ ]:
tst = jno.domain.load(f'/home/users/armbrust/projects/jno/examples/poseidon/inference_domain_290.pkl')
tst.context['interior_0'] = tst.context['interior']

In [ ]:
import jax
import equinox as eqx

def count_parameters(model_instance):
    """
    Count parameters in a core instance, properly handling LoRA fine-tuning.
    
    Returns:
        dict with 'total', 'trainable', and 'frozen' parameter counts
    """
    params = {}

    trainable_params = model_instance.training_logs[-1].get('trainable_params', None)
    frozen_params = model_instance.training_logs[-1].get('frozen_params', None)
    total_params = model_instance.training_logs[-1].get('total_params', None)
    lora_params = model_instance.training_logs[-1].get('lora_params', None)


    if trainable_params is None:
        
        for lid, model in model_instance.models.items():
            # Check if this is a LoRA-wrapped model (Flax style)
            from jno.architectures.lora_linear import FlaxLoRAWrapper, LoRALinear
            
            if isinstance(model, FlaxLoRAWrapper):
                # Flax LoRA wrapper: base_model is frozen, lora_params are trainable
                base_params = sum(
                    x.size for x in jax.tree_util.tree_leaves(model.base_model) 
                    if eqx.is_array(x)
                )
                lora_params = sum(
                    x.size for x in jax.tree_util.tree_leaves(model.lora_params) 
                    if eqx.is_array(x)
                )
                params[lid] = {
                    'total': base_params + lora_params,
                    'trainable': lora_params,
                    'frozen': base_params,
                    'is_lora': True
                }
            else:
                # Check for Equinox-style LoRA (LoRALinear layers)
                is_lora_layer = lambda x: isinstance(x, LoRALinear)
                lora_leaves = [
                    l for l in jax.tree_util.tree_leaves(model, is_leaf=is_lora_layer) 
                    if isinstance(l, LoRALinear)
                ]
                
                if lora_leaves:
                    # Has LoRA layers - count trainable (lora_A, lora_B) vs frozen (base weight)
                    trainable = 0
                    frozen = 0
                    
                    for leaf in jax.tree_util.tree_leaves(model, is_leaf=is_lora_layer):
                        if isinstance(leaf, LoRALinear):
                            # LoRA layer: lora_A and lora_B are trainable, weight is frozen
                            frozen += leaf.weight.size
                            if leaf.bias is not None:
                                frozen += leaf.bias.size
                            trainable += leaf.lora_A.size + leaf.lora_B.size
                        elif eqx.is_array(leaf):
                            # Other arrays - check if they're part of a LoRA layer
                            frozen += leaf.size
                    
                    params[lid] = {
                        'total': trainable + frozen,
                        'trainable': trainable,
                        'frozen': frozen,
                        'is_lora': True
                    }
                else:
                    # Standard model without LoRA
                    total = sum(
                        x.size for x in jax.tree_util.tree_leaves(model) 
                        if eqx.is_array(x)
                    )
                    params[lid] = {
                        'total': total,
                        'trainable': total,
                        'frozen': 0,
                        'is_lora': False
                    }
        

    else:
        params['total'] = total_params  
        params['trainable'] = trainable_params
        params['frozen'] = frozen_params
        params['is_lora'] = lora_params > 0
    
        params = {1: params}
    return params


# Usage example:
print(f"{'Name':<20} {'Total':>15} {'Trainable':>15} {'Frozen':>15} {'LoRA':>8}")
print("-" * 75)

def fn_parameters(models):
    params = {}
    for name, model in models.items():

        param_info = count_parameters(model)
        for lid, info in param_info.items():
            
            print(
                f"{name:>20}", 
                f"{info['total']:>15,} "
                f"{info['trainable']:>15,} "
                f"{info['frozen']:>15,} "
                f"{str(info['is_lora']):>8}"
            )
        # Store total trainable for your original dict format
        params[name] = info['trainable']
    return params

#params = fn_parameters(models)
lora_params = fn_parameters(models_lora)


In [ ]:
def fn_predictions(tst, models, pred=None):
    pred = {} if pred == None else pred
    pred['Source'] = tst.context["_f"][:240]
    pred['Ground Truth'] = tst.context["_u"][:240]

    for name, crux in models.items():
        if name not in pred.keys() or name == 'fno_both_256':
            pre = crux.eval(crux.constraints[0].args[0].right, tst)
            res_pre = pre.reshape((290, 1, 1, 128, 128, 1))
            pred[name] = res_pre[:240]
            print(f'{name} -> {pre.shape} -> {pred[name].shape}')
    return pred


#pred = fn_predictions(tst, models)
pred_lora = fn_predictions(tst, models_lora, pred_lora)

In [ ]:
p = models_lora['fno_both_256'].models[1](tst.context['_f'][0, 0, 0])

In [ ]:
plt.figure()
plt.imshow(p[...,0]*100)#pred_lora['fno_both_256'][1, 0, 0, :, :, 0]*200)
plt.colorbar()
plt.show()

In [ ]:
plt.figure()
plt.imshow(pred_lora['Ground Truth'][0, 0, 0, :, :, 0])
plt.colorbar()
plt.show()

In [ ]:
plt.figure()
plt.imshow(pred_lora['fno_data_256'][1, 0, 0, :, :, 0])
plt.colorbar()
plt.show()

In [ ]:

def fn_errors(pred):

    import numpy as np
    def error(u, _u):
        # L2 error (MSE)
        diff_l2 = np.square(u - _u)
        mean_diff_l2 = np.mean(diff_l2, axis=(1, 2, 3, 4, 5))
        
        # Relative L1 error: |u - _u| / |u|
        diff_l1 = np.abs(u - _u)
        abs_u = np.abs(u)
        
        # Avoid division by zero
        relative_l1 = diff_l1 / np.maximum(abs_u, 1e-8)
        median_relative_l1 = np.median(relative_l1, axis=(1, 2, 3, 4, 5))
        #median_relative_l1 = np.median(mean_relative_l1)
        
        return  median_relative_l1

    rmse = {}
    for name, crux in pred.items():
        if name not in ['Ground Truth', 'Source']:
            _rmse = error(pred[name], pred['Ground Truth'])
            rmse[name] = _rmse

    return rmse


#rmse = fn_errors(pred)
rmse_lora = fn_errors(pred_lora)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

idxs = [0, 1, 2, 3, 4, 5, 10, 20, 40, 80, 120, 180, 210, 230]

def plot(rmse, pred):
    min_rmse = np.min(rmse.values())
    fig, ax = plt.subplots(len(idxs), len(pred.keys()), figsize=(len(pred.keys())*1.2, len(idxs)*1.05), constrained_layout=False)
    fig.subplots_adjust(hspace=0.02, wspace=0.02)
    #ax = ax.ravel()
    for j, idx in enumerate(idxs):
        for i, (title, img) in enumerate(zip(pred.keys(), pred.values())):
            img = img[idx, 0, 0, ..., 0]
            gt = tst.context["_u"][idx, 0, 0, ..., 0]
            vmin = gt.min() 
            vmax = gt.max()
            if title != 'Source':
                im = ax[j, i].imshow(img, cmap="jet", vmin=vmin, vmax=vmax)
            else:
                im = ax[j, i].imshow(img, cmap="viridis")
            if title not in ['Ground Truth', 'Source']:
                if j == 0:
                    if rmse[title].mean() != min_rmse:
                        ax[j, i].set_title(f"{title}\n{rmse[title].mean():.4f} +/- {rmse[title].std():.3f}", fontsize=6)
                    else:
                        ax[j, i].set_title(f"{title}\n{rmse[title].mean():.4f} +/- {rmse[title].std():.3f}", fontsize=6, fontweight='bold')
            else:
                if j==0:
                    ax[j, i].set_title(title, fontsize=6)
            ax[j, i].axis("off")

    cbar = fig.colorbar(im, ax=ax, shrink=1.0, pad=0.01)
    plt.show()


#plot(rmse, pred)
plot(rmse_lora, pred_lora)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def fn_error_plot(params, rmse, names):
    """Horizontal violin plot - cleaner for comparing models."""
    
    # Sort by parameter count
    sorted_indices = np.argsort([params[n] for n in names])
    sorted_names = [names[i] for i in sorted_indices]
    
    # Simulate samples
    np.random.seed(42)
    data = []
    for n in sorted_names:
        data.append(rmse[n])
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.5)))
    
    # Create horizontal violin plots
    parts = ax.violinplot(data, positions=range(len(sorted_names)), 
                          vert=False, showmeans=True, showmedians=True)
    
    # Customize appearance
    for pc in parts['bodies']:
        pc.set_facecolor('steelblue')
        pc.set_alpha(0.7)
    parts['cmeans'].set_color('red')
    parts['cmedians'].set_color('orange')
    
    # Y-axis labels with parameter counts
    labels = [f"{n}\n({params[n]:,} params)" for n in sorted_names]
    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xscale('log')
    ax.set_xlabel("RMSE")
    ax.set_title("Model vs RMSE")
    ax.grid(True, axis='x', ls="--", alpha=0.4)
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='red', linewidth=2, label='Mean'),
        Line2D([0], [0], color='orange', linewidth=2, label='Median')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()

fn_error_plot(params, rmse, names)
fn_error_plot(lora_params, rmse_lora, names_lora)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

def fn_error_plot(rmse, names, title=None, figsize=(28, 12)):
    """
    Box plot with trajectories on x-axis.
    - Fill color = LoRA rank
    - Edge color = model type
    - Alpha/transparency = training type (data=solid, both=semi-transparent)

    """
    
    def parse_name(name):
        parts = name.split('_')
        if len(parts) > 3:
            lora_rank = int(parts[-1].replace('lora', ''))
        else:
            lora_rank = 0
        samples = int(parts[2])
        model_type = parts[0]
        training_type = parts[1]
        return model_type, training_type, samples, lora_rank
    
    parsed = {n: parse_name(n) for n in names}
    
    model_types = sorted(set(p[0] for p in parsed.values()))
    training_types = sorted(set(p[1] for p in parsed.values()))
    trajectories = sorted(set(p[2] for p in parsed.values()))
    lora_ranks = sorted(set(p[3] for p in parsed.values()))
    
    # Colors for LoRA ranks
    rank_colors = {
        0: 'gray',
        8: '#1f77b4',
        16: '#ff7f0e',
        32: '#2ca02c',
        64: '#d62728',
        128: '#9467bd',
    }
    
    # Edge styles for model types
    model_edges = {
        'poseidonT': {'color': 'black', 'linewidth': 1.5},
        'fno': {'color': 'red', 'linewidth': 2.5},
        'unet': {'color': 'purple', 'linewidth': 2.5},
        'deeponet': {'color': 'cyan', 'linewidth': 2.5},
    }
    
    # Alpha for training types
    training_alpha = {
        'data': 0.9,
        'both': 0.5,
    }
    
    # Hatching for training types (in addition to alpha)
    training_hatches = {
        'data': None,
        'both': '//',
    }
    
    fig, ax = plt.subplots(figsize=figsize)
    
    n_groups = len(lora_ranks) * len(model_types) * len(training_types)
    width = 0.75 / n_groups
    
    group_idx = 0
    for rank in lora_ranks:
        for model in model_types:
            for train_type in training_types:
                data = []
                positions = []
                
                for j, traj in enumerate(trajectories):
                    matching = [n for n in names if parsed[n] == (model, train_type, traj, rank)]
                    if matching:
                        data.append(rmse[matching[0]])
                        positions.append(j + (group_idx - n_groups/2 + 0.5) * width)
                
                if data:
                    fill_color = rank_colors.get(rank, 'gray')
                    edge_style = model_edges.get(model, {'color': 'black', 'linewidth': 1.5})
                    alpha = training_alpha.get(train_type, 0.75)
                    hatch = training_hatches.get(train_type, None)
                    
                    bp = ax.boxplot(
                        data, 
                        positions=positions, 
                        widths=width * 0.9,
                        patch_artist=True, 
                        showmeans=True,
                        medianprops=dict(color='white', linewidth=2),
                        meanprops=dict(marker='o', markerfacecolor='yellow', 
                                      markeredgecolor='black', markersize=4),
                        whiskerprops=dict(color=fill_color, linewidth=1.5),
                        capprops=dict(color=fill_color, linewidth=1.5),
                        flierprops=dict(marker='.', markerfacecolor=fill_color, 
                                       markersize=4, alpha=0.6)
                    )
                    
                    for patch in bp['boxes']:
                        patch.set_facecolor(fill_color)
                        patch.set_edgecolor(edge_style['color'])
                        patch.set_linewidth(edge_style['linewidth'])
                        patch.set_alpha(alpha)
                        if hatch:
                            patch.set_hatch(hatch)
                
                group_idx += 1
    
    # Formatting
    ax.set_xticks(range(len(trajectories)))
    ax.set_xticklabels(trajectories, fontsize=11)
    ax.set_xlabel("Number of Training Trajectories", fontsize=12)
    
    ax.set_yscale('log')
    ax.set_ylabel("RMSE (log scale)", fontsize=12)
    
    ax.set_title(title or "Model Error vs Training Trajectories", fontsize=14)
    ax.grid(True, axis='y', ls="--", alpha=0.4, which='both')
    ax.set_axisbelow(True)
    ax.set_xlim(-0.5, len(trajectories) - 0.5)
    
    # Legend - LoRA ranks
    legend_elements = [
        Patch(facecolor=rank_colors.get(r, 'gray'), alpha=0.75, 
              edgecolor='black', label=f'LoRA r={r}' if r > 0 else 'No LoRA') 
        for r in lora_ranks
    ]
    
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    
    # Legend - Model types
    for model in model_types:
        edge_style = model_edges.get(model, {'color': 'black', 'linewidth': 1.5})
        legend_elements.append(
            Patch(facecolor='white', edgecolor=edge_style['color'], 
                  linewidth=edge_style['linewidth'], label=f'{model}')
        )
    
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    
    # Legend - Training types
    for train_type in training_types:
        alpha = training_alpha.get(train_type, 0.75)
        hatch = training_hatches.get(train_type, None)
        legend_elements.append(
            Patch(facecolor='lightgray', alpha=alpha, edgecolor='black', 
                  hatch=hatch, label=f'Train: {train_type}')
        )
    
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    
    legend_elements.append(
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow',
                   markeredgecolor='black', markersize=6, label='Mean')
    )
    
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()


# Usage:

fn_error_plot(rmse_lora, names_lora)